In [ ]:
# Set custom jar
import os
import importlib.util
import sys
os.environ["CAPYMOA_MOA_JAR"] = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "custom_moa_full.jar"))

In [ ]:
# Import custom LogisticRegression model
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "src", "capymoa", "classifier", "_logistic_regression.py"))
spec = importlib.util.spec_from_file_location("capymoa.classifier._logistic_regression", file_path)
module = importlib.util.module_from_spec(spec)
sys.modules["capymoa.classifier._logistic_regression"] = module
spec.loader.exec_module(module)
LogisticRegression = module.LogisticRegression

In [ ]:
# Import custom SoftmaxRegression model
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "src", "capymoa", "classifier", "_softmax_regression.py"))
spec = importlib.util.spec_from_file_location("capymoa.classifier._softmax_regression", file_path)
module = importlib.util.module_from_spec(spec)
sys.modules["capymoa.classifier._softmax_regression"] = module
spec.loader.exec_module(module)
SoftmaxRegression = module.SoftmaxRegression

# 0. Getting started with CapyMOA

This notebook shows some basic usage of CapyMOA for supervised learning (classification and regression).

* There are more detailed notebooks and documentation available; our goal here is just to present some high-level functions and demonstrate a subset of CapyMOA's functionalities.
* For simplicity, we simulate data streams in the following examples using datasets and employing synthetic generators. One could also read data directly from a CSV or ARFF (See [stream_from_file](https://capymoa.org/api/modules/capymoa.stream.html#capymoa.stream.stream_from_file) function).

---

*More information about CapyMOA can be found at* https://www.capymoa.org

**last update on 28/11/2025**

## 0.1 Classification

* Classification for data streams traditionally assumes instances are available
  to the classifier in an incremental fashion and labels become available before
  a new instance becomes available.
* It is common to simulate this behavior using a **while loop**, often referred
  to as a **test-then-train loop** which contains 4 distinct steps:
    1. Fetches the next instance from the stream
    2. Makes a prediction
    3. Train the model with the instance
    4. Update a mechanism to keep track of metrics

**Some remarks about the test-then-train loop**:

* We must not train before testing, meaning that steps 2 and 3 should not be interchanged, as this would invalidate our interpretation concerning how the model performs on unseen data, leading to unreliable evaluations of its efficacy. 
* Steps 3 and 4 can be completed in any order without altering the result. 
* What if labels are not immediately available? Then you might want to read about delayed labeling and partially labeled data, see [A Survey on Semi-supervised Learning for Delayed Partially Labelled Data Streams](https://dl.acm.org/doi/full/10.1145/3523055)
* More information on classification for data streams is available at section **2.2 Classification** from the [Machine Learning for Data Streams](https://moa.cms.waikato.ac.nz/book-html/) book


In [ ]:
# This cell is hidden on capymoa.org. See docs/contributing/docs.rst
'''
from util.nbmock import mock_datasets, is_nb_fast

if is_nb_fast():
    mock_datasets()
'''

In [ ]:
from capymoa.datasets import Electricity
from capymoa.evaluation import ClassificationEvaluator
from capymoa.classifier import OnlineBagging

elec_stream = Electricity()

ob_learner = OnlineBagging(schema=elec_stream.get_schema(), ensemble_size=5)
ob_evaluator = ClassificationEvaluator(schema=elec_stream.get_schema())

log_reg_learner = LogisticRegression(schema=elec_stream.get_schema())
log_reg_evaluator = ClassificationEvaluator(schema=elec_stream.get_schema())

softmax_reg_learner = SoftmaxRegression(schema=elec_stream.get_schema())
softmax_reg_evaluator = ClassificationEvaluator(schema=elec_stream.get_schema())

for instance in elec_stream:
    ob_prediction = ob_learner.predict(instance)
    log_reg_prediction = log_reg_learner.predict(instance)
    softmax_reg_prediction = softmax_reg_learner.predict(instance)

    ob_learner.train(instance)
    log_reg_learner.train(instance)
    softmax_reg_learner.train(instance)

    ob_evaluator.update(instance.y_index, ob_prediction)
    log_reg_evaluator.update(instance.y_index, log_reg_prediction)
    softmax_reg_evaluator.update(instance.y_index, softmax_reg_prediction)

print(f"ob: {ob_evaluator.accuracy()}\n")
print(f"log_reg: {log_reg_evaluator.accuracy()}\n")
print(f"softmax_reg: {softmax_reg_evaluator.accuracy()}\n")

### 0.1.1 High-level evaluation functions

* If our goal is just to evaluate learners, it would be tedious to keep writing **test-then-train loops**. 
Thus, it makes sense to encapsulate that loop inside **high-level evaluation functions**. 

* Furthermore, sometimes we are interested in **cumulative metrics** and sometimes we care about **windowed metrics**. For example, if we want to know how accurate our model is so far, considering all the instances it has seen, then we would look at its **cumulative metrics**. However, we might also be interested in how well the model is performing every **n** number of instances, so that we can, for example, identify periods in which our model was really struggling to produce correct predictions. 

* In this example, we use the ```prequential_evaluation``` function, which provides us with both the cumulative and the windowed metrics! 

* Some remarks:
    * If you want to know more about other **high-level evaluation functions**, **evaluators**, or which **metrics** are available, check the **01_evaluation** notebook.
    * The **results** from evaluation functions such as **prequential_evaluation** follow a standard and are discussed thoroughly in the **Evaluation documentation** at http://www.capymoa.org.
    * Sometimes authors refer to the **cumulative** metrics as **test-then-train** metrics, such as **test-then-train accuracy** (or TTT accuracy for short). They all refer to the same concept.
    * Shouldn't we recreate the stream object ```elec_stream```? No, `prequential_evaluation()`, by default, will automatically ```restart()``` streams when they are reused.

In the below example `prequential_evaluation` is used with a `HoeffdingTree` classifier on the `Electricity` data stream.

In [ ]:
from capymoa.evaluation import prequential_evaluation
from capymoa.classifier import HoeffdingTree

ht = HoeffdingTree(schema=elec_stream.get_schema(), grace_period=50)

# Obtain the results from the high-level function.
# Note that we need to specify a window_size as we obtain both windowed and cumulative results.
# The results from a high-level evaluation function are represented as a PrequentialResults object.
results_ht = prequential_evaluation(stream=elec_stream, learner=ht, window_size=4500)

print(
    f"Cumulative accuracy = {results_ht.cumulative.accuracy()}, wall-clock time: {results_ht.wallclock()}"
)

# The windowed results are conveniently stored in a pandas DataFrame.
display(results_ht.windowed.metrics_per_window())

Duplicated cell for our LogisticRegression

In [ ]:
from capymoa.evaluation import prequential_evaluation
# from capymoa.classifier import HoeffdingTree

# ht = HoeffdingTree(schema=elec_stream.get_schema(), grace_period=50)
log_reg = LogisticRegression(
    schema=elec_stream.get_schema(),
)

# Obtain the results from the high-level function.
# Note that we need to specify a window_size as we obtain both windowed and cumulative results.
# The results from a high-level evaluation function are represented as a PrequentialResults object.
# results_ht = prequential_evaluation(stream=elec_stream, learner=ht, window_size=4500)
results_log_reg = prequential_evaluation(stream=elec_stream, learner=log_reg, window_size=4500)

print(
    f"Cumulative accuracy = {results_log_reg.cumulative.accuracy()}, wall-clock time: {results_log_reg.wallclock()}"
)

# The windowed results are conveniently stored in a pandas DataFrame.
display(results_log_reg.windowed.metrics_per_window())

Duplicated cell for our SoftmaxRegression

In [ ]:
from capymoa.evaluation import prequential_evaluation
# from capymoa.classifier import HoeffdingTree


# ht = HoeffdingTree(schema=elec_stream.get_schema(), grace_period=50)
softmax_reg = SoftmaxRegression(
    schema=elec_stream.get_schema(),
)

# Obtain the results from the high-level function.
# Note that we need to specify a window_size as we obtain both windowed and cumulative results.
# The results from a high-level evaluation function are represented as a PrequentialResults object.
# results_ht = prequential_evaluation(stream=elec_stream, learner=ht, window_size=4500)
results_softmax_reg = prequential_evaluation(stream=elec_stream, learner=softmax_reg, window_size=4500)

print(
    f"Cumulative accuracy = {results_softmax_reg.cumulative.accuracy()}, wall-clock time: {results_softmax_reg.wallclock()}"
)

# The windowed results are conveniently stored in a pandas DataFrame.
display(results_softmax_reg.windowed.metrics_per_window())

Brief comparison (not present in the original notebook)

In [ ]:
print(
    f"Cumulative accuracy - HoeffdingTree = {results_ht.cumulative.accuracy()}, wall-clock time: {results_ht.wallclock()}"
)

print(
    f"Cumulative accuracy - Logistic Regression = {results_log_reg.cumulative.accuracy()}, wall-clock time: {results_log_reg.wallclock()}"
)

print(
    f"Cumulative accuracy - Softmax Regression = {results_softmax_reg.cumulative.accuracy()}, wall-clock time: {results_softmax_reg.wallclock()}"
)

### 0.1.2 Comparing results among classifiers

* CapyMOA provides ```plot_windowed_results``` as an easy visualisation function for quickly comparing **windowed metrics**.
* In the example below, we create three classifiers: HoeffdingAdaptiveTree, HoeffdingTree and AdaptiveRandomForest, and plot the results using ```plot_windowed_results```.
* More details about ```plot_windowed_results``` options are described in the documentation at http://www.capymoa.org.

In [ ]:
from capymoa.evaluation.visualization import plot_windowed_results
from capymoa.base import MOAClassifier
from moa.classifiers.trees import HoeffdingAdaptiveTree
from capymoa.classifier import HoeffdingTree
from capymoa.classifier import AdaptiveRandomForestClassifier

# Create the wrapper for HoeffdingAdaptiveTree (from MOA).
HAT = MOAClassifier(
    schema=elec_stream.get_schema(), moa_learner=HoeffdingAdaptiveTree, CLI="-g 50"
)
HT = HoeffdingTree(schema=elec_stream.get_schema(), grace_period=50)
ARF = AdaptiveRandomForestClassifier(
    schema=elec_stream.get_schema(), ensemble_size=10, number_of_jobs=4
)
LOG_REG = LogisticRegression(
    schema=elec_stream.get_schema(),
) # default values for parameters
SOFTMAX_REG = SoftmaxRegression(
    schema=elec_stream.get_schema(),
) # default values for parameters

results_HAT = prequential_evaluation(stream=elec_stream, learner=HAT, window_size=4500)
results_HT = prequential_evaluation(stream=elec_stream, learner=HT, window_size=4500)
results_ARF = prequential_evaluation(stream=elec_stream, learner=ARF, window_size=4500)
results_LOG_REG = prequential_evaluation(stream=elec_stream, learner=LOG_REG, window_size=4500)
results_SOFTMAX_REG = prequential_evaluation(stream=elec_stream, learner=SOFTMAX_REG, window_size=4500)

# Comparing models based on their cumulative accuracy.
print(f"HAT accuracy = {results_HAT.cumulative.accuracy()}")
print(f"HT accuracy = {results_HT.cumulative.accuracy()}")
print(f"ARF accuracy = {results_ARF.cumulative.accuracy()}")
print(f"LOG_REG accuracy = {results_LOG_REG.cumulative.accuracy()}")
print(f"SOFTMAX_REG accuracy = {results_SOFTMAX_REG.cumulative.accuracy()}")

# Plotting the results. Note that we ovewrote the ylabel, but that doesn't change the metric.
plot_windowed_results(
    results_HAT,
    results_HT,
    results_ARF,
    results_LOG_REG,
    results_SOFTMAX_REG,
    metric="accuracy",
    xlabel="# Instances (window)",
)

## 0.2 Regression

* Regression algorithms have APIs very similar to classification algorithms. We can use the same high-level evaluation and visualisation functions for regression and classification.
* Similar to classification, we can also use MOA objects through a generic API.

In [ ]:
from capymoa.datasets import Fried
from moa.classifiers.trees import FIMTDD
from capymoa.base import MOARegressor
from capymoa.regressor import KNNRegressor

fried_stream = (
    Fried()
)  # Downloads the Fried dataset into the data dir in case it is not there yet.
fimtdd = MOARegressor(schema=fried_stream.get_schema(), moa_learner=FIMTDD())
knnreg = KNNRegressor(schema=fried_stream.get_schema(), k=3, window_size=1000)

results_fimtdd = prequential_evaluation(
    stream=fried_stream, learner=fimtdd, window_size=5000
)
results_knnreg = prequential_evaluation(
    stream=fried_stream, learner=knnreg, window_size=5000
)

results_fimtdd.windowed.metrics_per_window()
# Note that the metric is different from the ylabel parameter, which just overrides the y-axis label.
plot_windowed_results(
    results_fimtdd, results_knnreg, metric="rmse", ylabel="root mean squared error"
)

## 0.3 Concept drift

* One of the most challenging and defining aspects of data streams is the phenomenon known as **concept drifts**.
* In CapyMOA, we designed the simplest and most complete API for simulating, visualising and assessing concept drifts.
* In the example below, we focus on a simple way of simulating and visualising a drifting stream. There is a tutorial focusing entirely on how concept drift can be simulated, detected and assessed in a separate notebook (See **Tutorial 4**: `Simulating Concept Drifts with the DriftStream API`).

### 0.3.1 Plotting drift detection results

* This example uses the DriftStream building API, precisely the **positional version** where drifts are specified according to their exact location in the stream.
* **Integration with the visualisation function.** The DriftStream object carries meta-information about the drift which is passed along the stream and thus becomes available to ```plot_windowed_results```.

* The following plot contains two drifts: 1 abrupt and 1 gradual, such that the abrupt drift is located at instance 5000 and the gradual drift starts at instance 9000 and ends at 12000. This information is provided to the stream via ```GradualDrift(start=9000, end=12000)```.

* More details concerning concept drifts in CapyMOA can be found in the documentation at http://www.capymoa.org.

In [ ]:
from capymoa.classifier import OnlineBagging
from capymoa.stream.generator import SEA
from capymoa.stream.drift import AbruptDrift, GradualDrift, DriftStream

# Generating a synthetic stream with 1 abrupt drift and 1 gradual drift.
stream_sea2drift = DriftStream(
    stream=[
        SEA(function=1),
        AbruptDrift(position=5000),
        SEA(function=3),
        GradualDrift(start=9000, end=12000),
        SEA(function=1),
    ]
)

OB = OnlineBagging(schema=stream_sea2drift.get_schema(), ensemble_size=10)

# Since this is a synthetic stream, max_instances is needed to determine the amount of instances to be generated.
results_sea2drift_OB = prequential_evaluation(
    stream=stream_sea2drift, learner=OB, window_size=100, max_instances=15000
)

plot_windowed_results(results_sea2drift_OB, metric="accuracy")

Duplicated cell for our LogisticRegression

In [ ]:
# from capymoa.classifier import OnlineBagging
from capymoa.stream.generator import SEA
from capymoa.stream.drift import AbruptDrift, GradualDrift, DriftStream

# Generating a synthetic stream with 1 abrupt drift and 1 gradual drift.
stream_sea2drift = DriftStream(
    stream=[
        SEA(function=1),
        AbruptDrift(position=5000),
        SEA(function=3),
        GradualDrift(start=9000, end=12000),
        SEA(function=1),
    ]
)

# OB = OnlineBagging(schema=stream_sea2drift.get_schema(), ensemble_size=10)
LOG_REG = LogisticRegression(schema=stream_sea2drift.get_schema())

# Since this is a synthetic stream, max_instances is needed to determine the amount of instances to be generated.
'''
results_sea2drift_OB = prequential_evaluation(
    stream=stream_sea2drift, learner=OB, window_size=100, max_instances=15000
)
'''
results_sea2drift_LOG_REG = prequential_evaluation(
    stream=stream_sea2drift, learner=LOG_REG, window_size=100, max_instances=15000
)

plot_windowed_results(results_sea2drift_LOG_REG, metric="accuracy")

Duplicated cell for our SoftmaxRegression

In [ ]:
# from capymoa.classifier import OnlineBagging
from capymoa.stream.generator import SEA
from capymoa.stream.drift import AbruptDrift, GradualDrift, DriftStream

# Generating a synthetic stream with 1 abrupt drift and 1 gradual drift.
stream_sea2drift = DriftStream(
    stream=[
        SEA(function=1),
        AbruptDrift(position=5000),
        SEA(function=3),
        GradualDrift(start=9000, end=12000),
        SEA(function=1),
    ]
)

# OB = OnlineBagging(schema=stream_sea2drift.get_schema(), ensemble_size=10)
SOFTMAX_REG = SoftmaxRegression(schema=stream_sea2drift.get_schema())

# Since this is a synthetic stream, max_instances is needed to determine the amount of instances to be generated.
'''
results_sea2drift_OB = prequential_evaluation(
    stream=stream_sea2drift, learner=OB, window_size=100, max_instances=15000
)
'''
results_sea2drift_SOFTMAX_REG = prequential_evaluation(
    stream=stream_sea2drift, learner=SOFTMAX_REG, window_size=100, max_instances=15000
)

plot_windowed_results(results_sea2drift_SOFTMAX_REG, metric="accuracy")

Brief comparison (not present in the original notebook)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metric = "accuracy"
window_size = 100

df_ob = results_sea2drift_OB.windowed.metrics_per_window()
df_lr = results_sea2drift_LOG_REG.windowed.metrics_per_window()
df_sm = results_sea2drift_SOFTMAX_REG.windowed.metrics_per_window()

x_ob = np.arange(1, len(df_ob) + 1) * window_size
x_lr = np.arange(1, len(df_lr) + 1) * window_size
x_sm = np.arange(1, len(df_sm) + 1) * window_size

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, x, df, title in [
    (axes[0], x_ob, df_ob, "Online Bagging"),
    (axes[1], x_lr, df_lr, "Logistic Regression"),
    (axes[2], x_sm, df_sm, "Softmax Regression"),
]:
    ax.plot(x, df[metric], marker="o", markersize=3)
    ax.set_title(title)
    ax.set_xlabel("# Instances")

    # drift abrupt
    ax.axvline(5000, color="red")

    # drift gradual
    ax.axvspan(9000, 12000, color="red", alpha=0.2)

axes[0].set_ylabel("Accuracy")

plt.tight_layout()
plt.show()